In [5]:
import os
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
import scanpy as sc

%matplotlib inline

In [6]:
## Import In Disease Subtype Level Stats ##

CSV_PATH = '/mnt/sdb/scz_meta_analysis_processed/dge_signatures/dreamlet_dges/disease_analyses/'

vp_lst = pd.read_csv(CSV_PATH + "/variance_partition_long_subtype.csv")

dge_22q11 = pd.read_csv(CSV_PATH + "dreamlet_22q11_dge_scz_results_subtype.csv")
dge_NRXN1 = pd.read_csv(CSV_PATH + "dreamlet_NRNXN1_dge_scz_results_subtype.csv")
dge_3q29 = pd.read_csv(CSV_PATH + "dreamlet_3q29_dge_scz_results_subtype.csv")
dge_15q13 = pd.read_csv(CSV_PATH + "dreamlet_15q13_dge_scz_results_subtype.csv")
dge_Idiopathic = pd.read_csv(CSV_PATH + "dreamlet_Idiopathic_dge_scz_results_subtype.csv")

In [19]:
dge_22q11['significant'] = (dge_22q11['adj.P.Val'] < 0.05)
dge_NRXN1['significant'] = (dge_NRXN1['adj.P.Val'] < 0.05)
dge_3q29['significant'] = (dge_3q29['adj.P.Val'] < 0.05)
dge_15q13['significant'] = (dge_15q13['adj.P.Val'] < 0.05)
dge_Idiopathic['significant'] = (dge_Idiopathic['adj.P.Val'] < 0.05)

In [39]:
dge_Idiopathic

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
0,Stressed RG FTL+EIF1+GDF15+,LYPLA1,3.137826,5.651985,11.659565,2.226344e-19,5.505193e-14,33.561956,9.001510,True
1,Ventral RG GAD2+S100B+PDGFRA+,ALAD,3.677798,3.300585,11.236389,9.281155e-19,1.147499e-13,32.036676,8.843445,True
2,Ventral RG GAD2+S100B+PDGFRA+,CMSS1,2.360015,6.344260,12.171739,1.565326e-14,1.290220e-09,27.930729,7.682075,True
3,Ventral RG GAD2+S100B+PDGFRA+,FAM136A,2.308896,6.205123,9.621806,5.925008e-14,2.599997e-09,23.956063,7.509729,True
4,Ventral RG GAD2+S100B+PDGFRA+,C14orf132,-4.113323,5.494729,-8.913970,6.331913e-14,2.599997e-09,18.527215,-7.501028,True
...,...,...,...,...,...,...,...,...,...,...
247270,Glioblast PTGDS+CA12+EDNRB+,ZNF680,-0.000021,4.599911,-0.000019,9.999849e-01,9.999957e-01,-4.613330,-0.000019,False
247271,Proliferating Dorsal RG E2F1+PCNA+MCM5+,CIR1,0.000008,5.908088,0.000017,9.999861e-01,9.999957e-01,-4.996652,0.000017,False
247272,S-Phase Midbrain/Hindbrain RG SOX3+H4C9+EN2+,PANK1,0.000005,3.928811,0.000016,9.999876e-01,9.999957e-01,-4.823463,0.000016,False
247273,Skin/Muscle-Like Cells KRT8+KRT18+TNNT1+,MRPS26,0.000002,6.699538,0.000006,9.999953e-01,9.999990e-01,-5.676404,0.000006,False


In [35]:
dge = pd.concat([
                dge_22q11.assign(group="22q11"),
                dge_NRXN1.assign(group="NRXN1"),
                dge_3q29.assign(group="3q29"),
                dge_15q13.assign(group="15q13"),
                dge_Idiopathic.assign(group="Idiopathic")],
    ignore_index=True)

In [37]:
sns.scatterplot(dge[dge['significant']], x='group', hue='assay', y='logFC')

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant,group
0,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,RTN4R,-0.905185,4.577162,-7.170698,3.093560e-09,0.000359,7.843889,-5.926555,True,22q11
1,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,MED15,-0.879371,4.509442,-7.162167,3.994192e-09,0.000359,5.996244,-5.884434,True,22q11
2,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,CRKL,-0.997855,4.364135,-7.180986,4.261523e-09,0.000359,7.413839,-5.873709,True,22q11
3,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,PI4KA,-0.875372,5.327724,-7.626761,6.087131e-09,0.000359,8.324293,-5.814346,True,22q11
4,M-Phase RG CCNB1+CDC20+PLK1+,C22orf39,-0.811150,6.035838,-6.889304,6.347249e-09,0.000359,1.278814,-5.807342,True,22q11
...,...,...,...,...,...,...,...,...,...,...,...
1116922,S-Phase RG TYMS+GINS2+PCLAF+,BCL7A,-2.053703,4.926098,-3.485762,8.733148e-04,0.049953,-0.943620,-3.328447,True,Idiopathic
1116923,Ventral RG GAD2+S100B+PDGFRA+,PDP1,-1.798940,3.726187,-3.536816,8.736747e-04,0.049963,-1.009187,-3.328333,True,Idiopathic
1116924,M-Phase RG CCNB1+CDC20+PLK1+,MRS2,1.240840,3.870361,3.829597,8.740508e-04,0.049972,0.071716,3.328213,True,Idiopathic
1116925,Upper Layer Neuron Transitional SATB2+CUX2+DLX1+,FGD1,2.518765,4.720632,3.464210,8.744833e-04,0.049979,-0.746210,3.328075,True,Idiopathic


In [34]:
dge_22q11[dge_22q11['adj.P.Val'] < 0.05][['assay', 'ID']]

,assay,ID
0,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,RTN4R
1,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,MED15
2,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,CRKL
3,Deep Layer Neuron BCL11B+TBR1+GRIN2B+,PI4KA
4,M-Phase RG CCNB1+CDC20+PLK1+,C22orf39
...,...,...
111,Unknown Neuron CALY+MEF2C+CNTN1+,RANBP1
112,Choroid Plexus Primed RG TTR+RSPO2+OTX2+,MRPL40
113,Newborn Neuron NEUROG1+NEUROD1+EBF1+,UFD1
114,S-Phase Midbrain/Hindbrain RG SOX3+H4C9+EN2+,DGCR2


In [12]:
dge_NRXN1[dge_NRXN1['adj.P.Val'] < 0.05]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std


In [10]:
dge_3q29[dge_3q29['adj.P.Val'] < 0.05]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
0,Mature MGE Interneuron SIX3+TLE4+ZFHX4+,CHCHD2,11.269012,6.540623,7.654655,3.872759e-09,0.001086,-3.239123,5.889538
1,Fibroblast PDGFRB+COL6A1+COL1A2+,CTSF,6.706717,3.308132,9.506924,8.301046e-09,0.001119,-1.281564,5.762229
2,Neurogenic RG SIX1+DCX+STMN2+,MIR9-1HG,-42.995502,5.251674,-8.112360,1.197362e-08,0.001119,1.131407,-5.700101
3,M-Phase RG CCNB1+CDC20+PLK1+,BLOC1S6,-1.344280,5.910608,-6.511956,2.352653e-08,0.001214,-0.621134,-5.583840
4,Ventral RG GAD2+S100B+PDGFRA+,BLOC1S6,-0.944248,6.081971,-6.231410,2.484082e-08,0.001214,6.159880,-5.574383
...,...,...,...,...,...,...,...,...,...
140,RG FABP7+PTPRZ1+EGFR+,COL6A1,-2.520100,5.322123,-4.730320,2.483573e-05,0.049077,0.154465,-4.216287
141,Neurogenic RG SIX1+DCX+STMN2+,CCND2,15.860596,7.083504,4.726594,2.485732e-05,0.049077,1.015850,4.216091
142,Schwann Cell S100B+SOX10+PLP1+,TOP2A,6.085770,5.831024,4.790693,2.530616e-05,0.049614,2.956182,4.212052
143,M-Phase RG CCNB1+CDC20+PLK1+,FGF11,-4.658277,1.932567,-4.468233,2.564833e-05,0.049828,-4.257736,-4.209018


In [13]:
dge_15q13[dge_15q13['adj.P.Val'] < 0.05]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
0,Hindbrain/Cerebellar EN CNTNAP2+INSM1+PRPH+,TCEAL5,3.260124,5.495842,8.025112,8.592696e-11,0.000023,-1.042682,6.489841
1,Upper Layer Neuron Transitional SATB2+CUX2+DLX1+,TCEAL5,4.025703,5.365793,7.780503,3.055794e-10,0.000027,-1.964511,6.295921
2,Unknown Neuron CALY+MEF2C+CNTN1+,BEX5,6.172647,6.497098,7.478295,3.062814e-10,0.000027,0.100586,6.295565
3,Stressed Pons/Medulla Neuron DDIT3+ATF3+PMAIP1+,BEX5,3.922626,6.407434,6.798969,6.254277e-09,0.000415,3.630747,5.809813
4,CP-Primed RG TTR+FABP7+HOPX+,TCEAL5,3.289904,4.418154,6.453331,2.518935e-08,0.001336,5.258936,5.571957
5,Stressed Pons/Medulla Neuron DDIT3+ATF3+PMAIP1+,TCEAL5,3.794289,5.399309,5.919900,1.515962e-07,0.006698,3.060387,5.250610
6,Ventral RG GAD2+S100B+PDGFRA+,BEX5,4.207299,3.417658,5.651737,4.294209e-07,0.016264,-3.552498,5.055431
7,Choroid Plexus Primed RG TTR+RSPO2+OTX2+,METTL16,3.854518,5.017541,5.436277,5.372654e-07,0.017805,-3.889181,5.012503
8,Hypothalamic EN TLX3+IRX3+PAX3+,ABHD10,2.358165,5.047194,5.365379,8.941214e-07,0.026338,4.758042,4.913615
9,Stressed Neuron CDKN1A+PMAIP1+TNFRSF12A+,BEX5,4.582771,6.329955,5.277112,1.297225e-06,0.034391,1.902889,4.840179


In [18]:
dge_Idiopathic[dge_Idiopathic['adj.P.Val'] < 0.05]

,assay,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std
0,Stressed RG FTL+EIF1+GDF15+,LYPLA1,3.137826,5.651985,11.659565,2.226344e-19,5.505193e-14,33.561956,9.001510
1,Ventral RG GAD2+S100B+PDGFRA+,ALAD,3.677798,3.300585,11.236389,9.281155e-19,1.147499e-13,32.036676,8.843445
2,Ventral RG GAD2+S100B+PDGFRA+,CMSS1,2.360015,6.344260,12.171739,1.565326e-14,1.290220e-09,27.930729,7.682075
3,Ventral RG GAD2+S100B+PDGFRA+,FAM136A,2.308896,6.205123,9.621806,5.925008e-14,2.599997e-09,23.956063,7.509729
4,Ventral RG GAD2+S100B+PDGFRA+,C14orf132,-4.113323,5.494729,-8.913970,6.331913e-14,2.599997e-09,18.527215,-7.501028
...,...,...,...,...,...,...,...,...,...
4322,S-Phase RG TYMS+GINS2+PCLAF+,BCL7A,-2.053703,4.926098,-3.485762,8.733148e-04,4.995349e-02,-0.943620,-3.328447
4323,Ventral RG GAD2+S100B+PDGFRA+,PDP1,-1.798940,3.726187,-3.536816,8.736747e-04,4.996251e-02,-1.009187,-3.328333
4324,M-Phase RG CCNB1+CDC20+PLK1+,MRS2,1.240840,3.870361,3.829597,8.740508e-04,4.997246e-02,0.071716,3.328213
4325,Upper Layer Neuron Transitional SATB2+CUX2+DLX1+,FGD1,2.518765,4.720632,3.464210,8.744833e-04,4.997908e-02,-0.746210,3.328075


In [28]:
dge_22q11[dge_22q11['assay'].str.contains('Str') & dge_22q11['significant']].groupby('assay').nunique()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
assay,,,,,,,,,
Stressed Pons/Medulla Neuron DDIT3+ATF3+PMAIP1+,1,1,1,1,1,1,1,1,1


In [29]:
dge_NRXN1[dge_NRXN1['assay'].str.contains('Str') & dge_NRXN1['significant']].groupby('assay').nunique()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
assay,,,,,,,,,


In [30]:
dge_3q29[dge_3q29['assay'].str.contains('Str') & dge_3q29['significant']].groupby('assay').nunique()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
assay,,,,,,,,,
Stressed RG FTL+EIF1+GDF15+,5,5,5,5,5,5,5,5,1


In [31]:
dge_15q13[dge_15q13['assay'].str.contains('Str') & dge_15q13['significant']].groupby('assay').nunique()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
assay,,,,,,,,,
Stressed Neuron CDKN1A+PMAIP1+TNFRSF12A+,1,1,1,1,1,1,1,1,1
Stressed Pons/Medulla Neuron DDIT3+ATF3+PMAIP1+,2,2,2,2,2,2,2,2,1


In [32]:
dge_Idiopathic[dge_Idiopathic['assay'].str.contains('Str') & dge_Idiopathic['significant']].groupby('assay').nunique()

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B,z.std,significant
assay,,,,,,,,,
Stressed Neuron CDKN1A+PMAIP1+TNFRSF12A+,65,65,65,65,65,62,65,65,1
Stressed Pons/Medulla Neuron DDIT3+ATF3+PMAIP1+,52,52,52,52,52,52,52,52,1
Stressed RG FTL+EIF1+GDF15+,838,838,838,838,838,729,838,838,1
